In [0]:
# Upgrade Databricks SDK to the latest version and restart Python to see updated packages
%pip install --upgrade databricks-sdk==0.70.0
%restart_python

from databricks.sdk.service.jobs import JobSettings as Job


Budget_Analysis = Job.from_dict(
    {
        "name": "Budget_Analysis",
        "description": "E2E Budget Analysis Pipeline used for Controlling the Budgets ",
        "email_notifications": {
            "on_failure": [
                "shogun.banik.jobs.09@gmail.com",
            ],
        },
        "schedule": {
            "quartz_cron_expression": "0 45 20 ? * SAT,SUN *",
            "timezone_id": "Asia/Calcutta",
            "pause_status": "UNPAUSED",
        },
        "max_concurrent_runs": 4,
        "tasks": [
            {
                "task_key": "CommonEnv",
                "notebook_task": {
                    "notebook_path": "/Workspace/Budgetify/Common_Env",
                    "source": "WORKSPACE",
                },
            },
            {
                "task_key": "Extraction_Module_Excel2CSV",
                "depends_on": [
                    {
                        "task_key": "CommonEnv",
                    },
                ],
                "notebook_task": {
                    "notebook_path": "/Workspace/Budgetify/Python_extraction_csv",
                    "source": "WORKSPACE",
                },
                "max_retries": 1,
                "min_retry_interval_millis": 0,
                "disable_auto_optimization": True,
            },
            {
                "task_key": "Monthly_Analysis_LoopedIn",
                "depends_on": [
                    {
                        "task_key": "Extraction_Module_Excel2CSV",
                    },
                ],
                "for_each_task": {
                    "inputs": "[\"april\",\"may\",\"june\",\"july\",\"august\",\"september\", \"october\",\"november\",\"december\",\"january\",\"february\",\"march\"]",
                    "concurrency": 3,
                    "task": {
                        "task_key": "Monthly_Analysis_LoopedIn_iteration",
                        "notebook_task": {
                            "notebook_path": "/Workspace/Budgetify/Monthly_Analysis_Refined",
                            "base_parameters": {
                                "month": "{{input}}",
                            },
                            "source": "WORKSPACE",
                        },
                        "max_retries": 1,
                        "min_retry_interval_millis": 0,
                        "disable_auto_optimization": True,
                    },
                },
            },
            {
                "task_key": "Budget_BI_Analytics",
                "depends_on": [
                    {
                        "task_key": "Monthly_Analysis_LoopedIn",
                    },
                ],
                "dashboard_task": {
                    "subscription": {
                        "subscribers": [
                            {
                                "user_name": "shogun.banik.jobs.09@gmail.com",
                            },
                        ],
                        "custom_subject": "Monthly Reports",
                    },
                    "warehouse_id": "8380cb799766b313",
                    "dashboard_id": "01f1305b18f515d4b166cfb8e5731e01",
                },
                "max_retries": 1,
                "min_retry_interval_millis": 0,
                "email_notifications": {
                    "on_failure": [
                        "shogun.banik.jobs.09@gmail.com",
                    ],
                },
            },
            {
                "task_key": "Quaterly_Analysis",
                "depends_on": [
                    {
                        "task_key": "Monthly_Analysis_LoopedIn",
                    },
                ],
                "notebook_task": {
                    "notebook_path": "/Workspace/Budgetify/Quaterly_Analysis",
                    "source": "WORKSPACE",
                },
            },
            {
                "task_key": "Quaterly_Reports",
                "depends_on": [
                    {
                        "task_key": "Quaterly_Analysis",
                    },
                    {
                        "task_key": "Budget_BI_Analytics",
                    },
                ],
                "dashboard_task": {
                    "subscription": {
                        "subscribers": [
                            {
                                "user_name": "shogun.banik.jobs.09@gmail.com",
                            },
                        ],
                        "custom_subject": "Quaterly Reports",
                    },
                    "warehouse_id": "8380cb799766b313",
                    "dashboard_id": "01f1305f4ffd1cf6a0ac81cff2618094",
                },
            },
        ],
        "queue": {
            "enabled": True,
        },
        "performance_target": "PERFORMANCE_OPTIMIZED",
    }
)

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
w.jobs.reset(new_settings=Budget_Analysis, job_id=651107726631341)
# or create a new job using: w.jobs.create(**Budget_Analysis.as_shallow_dict())
